In [1]:
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.api import VAR
import pandas as pd
import numpy as np

**Setting up the dataset into a pandas dataframe**

In [2]:
df = pd.read_csv("../../data/FRB_H15.csv").dropna()

#rename columns
df.rename(columns={"Series Description": "Date", "Market yield on U.S. Treasury securities at 1-month  constant maturity, quoted on investment basis": "0Y1M", "Market yield on U.S. Treasury securities at 3-month  constant maturity, quoted on investment basis": "0Y3M", "Market yield on U.S. Treasury securities at 6-month  constant maturity, quoted on investment basis": "0Y6M", "Market yield on U.S. Treasury securities at 1-year  constant maturity, quoted on investment basis": "1Y", "Market yield on U.S. Treasury securities at 2-year  constant maturity, quoted on investment basis": "2Y", "Market yield on U.S. Treasury securities at 3-year  constant maturity, quoted on investment basis": "3Y", "Market yield on U.S. Treasury securities at 5-year  constant maturity, quoted on investment basis": "5Y", "Market yield on U.S. Treasury securities at 7-year  constant maturity, quoted on investment basis": "7Y", "Market yield on U.S. Treasury securities at 10-year  constant maturity, quoted on investment basis": "10Y", "Market yield on U.S. Treasury securities at 20-year constant maturity, quoted on investment basis": "20Y", "Market yield on U.S. Treasury securities at 30-year  constant maturity, quoted on investment basis": "30Y"}, inplace=True)
df.rename(columns={"Market yield on U.S. Treasury securities at 20-year  constant maturity, quoted on investment basis": "20Y"}, inplace=True)

#make index to datetime for timeseries
df["Date"] = pd.to_datetime(df["Date"])
df.set_index("Date", inplace=True)

df.tail(20)

,0Y1M,0Y3M,0Y6M,1Y,2Y,3Y,5Y,7Y,10Y,20Y,30Y
Date,,,,,,,,,,,
2026-04-17,3.69,3.70,3.69,3.64,3.71,3.72,3.84,4.04,4.26,4.85,4.88
2026-04-20,3.69,3.71,3.72,3.65,3.72,3.73,3.86,4.04,4.26,4.85,4.88
2026-04-21,3.67,3.69,3.73,3.69,3.78,3.80,3.91,4.09,4.30,4.87,4.89
2026-04-22,3.68,3.69,3.72,3.69,3.79,3.81,3.91,4.10,4.30,4.87,4.90
2026-04-23,3.69,3.69,3.72,3.70,3.83,3.84,3.96,4.13,4.34,4.90,4.92
2026-04-24,3.69,3.69,3.71,3.67,3.78,3.80,3.92,4.10,4.31,4.88,4.91
2026-04-27,3.70,3.68,3.72,3.69,3.78,3.83,3.94,4.14,4.35,4.92,4.94
2026-04-28,3.68,3.68,3.72,3.71,3.84,3.86,3.97,4.16,4.36,4.92,4.94
2026-04-29,3.68,3.68,3.73,3.75,3.92,3.94,4.05,4.23,4.42,4.97,4.98


****
**Training AR models to see differences between lag values**

In [3]:
#split the data into different maturities (for AR model)

trainSplit = {}
#maturities_list = []
for col in df.columns:
    #maturities[col] = df[[col]]
    #maturities_list.append(df[[col]])

    curr = df[[col]]
    trainSplit[col] = {"train": curr[curr.index <= "2025-4-17"], "test": curr[curr.index > "2025-4-17"]} 

    #removes date as indexed column for training data so statsmodels doesnt complain
    trainSplit[col]["train"] = trainSplit[col]["train"].reset_index(drop=True)

#create an AR model for each maturity

modelArr = []

#testing different lag numbers for ideal rmse

matList = ["0Y1M", "0Y3M", "0Y6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]
for i in range(len(matList)):
    modelArr.append([])

j=0
for mat in matList:
    
    for i in range(1,31):
        modelArr[j].append( 
            AutoReg(
                endog = trainSplit[mat]["train"][mat],
                lags = i,
                trend = "c"
            )
        )
    j+=1

resultsArr = []
for i in range(len(matList)):
    resultsArr.append([])

i = 0
for arr in modelArr:

    for model in arr:
        resultsArr[i].append(model.fit())
    i+=1


# adds 30 models with lags from 1 - 30 for each maturity to resultsArr
# the index from resultsArr is the index in matList, and index of inner array is lag - 1

****
**Forecasting and calculating error values for each lag value (aggregated results)**

In [4]:
#test the model based on y_test

#resultsArr - each arr is a maturity, inner arr is the lags
#trainSplit - key is maturity, from matList, inner dict has train and test dataframes


#forecast maturities so you can compare to actual
forecastArr = []


for i in range(len(resultsArr)):
    forecastArr.append([])
    for result in resultsArr[i]:
        forecastArr[i].append(result.forecast(steps=20).to_numpy())

testArr = []
for mat in matList:
    testArr.append(trainSplit[mat]["test"][mat].to_numpy())

#predicted = forecast.to_numpy()

#primary array - each maturity, inner array is error for each given lag
errorsArr = []
errors1520 = []

i=0
for mat in forecastArr:
    errorsArr.append([])
    errors1520.append([])

    for j in range(len(mat)):

        #errorsArr[i] is the array for one maturity, errorsArr[i][j] is the error for a given lag
        error_1day = forecastArr[i][j][0] - testArr[i][0]
        error_5day = forecastArr[i][j][4] - testArr[i][4]
        error_20day = forecastArr[i][j][19] - testArr[i][19]

        errorsArr[i].append(forecastArr[i][j] - testArr[i][:20])

        #first value is maturity, then is the lag value, then error for 1,5, 20 day forecast
        # errors1520[0][0][0] = error for 1 day forecast for 0Y1M with lag 1
        errors1520[i].append([error_1day, error_5day, error_20day])

    i+=1


rmse1520 = []
rmseArr = []
avgRMSE = []

for x in range(len(matList)): 
    rmseArr.append([])
    rmse1520.append([])
    avgRMSE.append([]) 

for i in range(len(errorsArr)):
    for j in range(len(errorsArr[i])):
        rmseArr[i].append(np.sqrt(np.mean(errorsArr[i][j]**2)))

        #stores rmse for 1, 5, and 20 day forecasts for each maturity and lag

        rmse_1day = np.sqrt(np.mean(errors1520[i][j][0]**2))
        rmse_5day = np.sqrt(np.mean(errors1520[i][j][1]**2))
        rmse_20day = np.sqrt(np.mean(errors1520[i][j][2]**2))

        #, (rmse_1day + rmse_5day + rmse_20day) / 3

        rmse1520[i].append([rmse_1day, rmse_5day, rmse_20day])
        avgRMSE[i].append([(rmse_1day + rmse_5day + rmse_20day) / 3])

    # print("Maturity: ", matList[i], " RMSE 1, 5, 20 day: ", rmse1520[i])


'''
for i in range(len(rmse1520)):
    for j in range(len(rmse1520[i])):
        # print("maturity: ", matList[i], " lag: ", j+1, " RMSE 1, 5, 20 day: ", rmse1520[i][j])
        print ("maturity: ", matList[i], " lag: ", j+1, " RMSE 1, 5, 20 day: ", rmse1520[i][j], " avg RMSE: ", avgRMSE[i][j])




        
lags = [[[] for j in range(30)] for i in range(11)]
for i in range(11):
    for j in range(30):
        #add the rmse for each maturity and lag to the corresponding lag index in lags
        #lags[i][j] where i is the maturity index, j is the lag index, value is the error for that given combo of ONLY 1,5,20 days
        lags[i][j].append( sum([rmse1520[i][j][0], rmse1520[i][j][1], rmse1520[i][j][2]]) / 3 )

        #output those values per lag for only 1,5,20 day forecasts
        print("maturity: ", matList[i], " lag: ", j+1, " RMSE 1, 5, 20 day: ", lags[i][j])
'''

aggregatedError_bylag_1520 = {}
for i in range(1,31):
    # i tracks lag
    aggregatedError_bylag_1520[i] = np.array([0.0, 0.0, 0.0])

    for j in range(len(matList)):
        # j tracks maturity
        aggregatedError_bylag_1520[i] += np.array(avgRMSE[j][i-1])

    aggregatedError_bylag_1520[i] = aggregatedError_bylag_1520[i] / len(matList)


aggregatedError_bylag = {}
# (dictionary so we can sort the values easily)
for i in range(1,31):
    aggregatedError_bylag[i] = np.array([0.0, 0.0, 0.0])

    for j in range(len(matList)):
        aggregatedError_bylag[i] += np.array(rmse1520[j][i-1])

    aggregatedError_bylag[i] = aggregatedError_bylag[i] / len(matList)


'''
for key,value in sorted(aggregatedError_bylag.items(), key = lambda item: item[1][0]):
    print(key, " : ", value)
'''
for key,value in sorted(aggregatedError_bylag_1520.items(), key = lambda item: item[1][0]):
    print(key, " : ", value)


# above output showcases that different lag values dont really have an impact on the RMSE for 1, 5, and 20 day forecasts. 
# The RMSE is relatively stable across different lag values, indicating that the choice of lag may not significantly affect the model's predictive performance for these specific forecast horizons.
# a simple decision made is using an AR of 5 days, which is one trading week (5 days). 
# Yield curve data is provided on a daily basis, and only captures monday through friday, so a lag of 5 is one week



19  :  [0.07011398 0.07011398 0.07011398]
15  :  [0.0701355 0.0701355 0.0701355]
23  :  [0.07018227 0.07018227 0.07018227]
18  :  [0.0703236 0.0703236 0.0703236]
5  :  [0.07051766 0.07051766 0.07051766]
20  :  [0.07060877 0.07060877 0.07060877]
4  :  [0.07068152 0.07068152 0.07068152]
16  :  [0.07068375 0.07068375 0.07068375]
7  :  [0.0706899 0.0706899 0.0706899]
17  :  [0.07075218 0.07075218 0.07075218]
24  :  [0.07083554 0.07083554 0.07083554]
6  :  [0.07093285 0.07093285 0.07093285]
3  :  [0.07096654 0.07096654 0.07096654]
21  :  [0.07113084 0.07113084 0.07113084]
22  :  [0.0711311 0.0711311 0.0711311]
25  :  [0.071289 0.071289 0.071289]
1  :  [0.07135059 0.07135059 0.07135059]
14  :  [0.07155681 0.07155681 0.07155681]
8  :  [0.07157028 0.07157028 0.07157028]
9  :  [0.0716225 0.0716225 0.0716225]
26  :  [0.07166562 0.07166562 0.07166562]
27  :  [0.07167985 0.07167985 0.07167985]
2  :  [0.07174356 0.07174356 0.07174356]
13  :  [0.0719776 0.0719776 0.0719776]
10  :  [0.0721799 0.07217

**** 
***Final AR model training, RMSE calculations***

In [5]:
#creating a baseline AR model with lag of 5 days for each maturity, calculating RMSE for 1,5, and 20 day forecasts, and comparing to actual values

models = []
# trainSplit dict
# matList = ["0Y1M", "0Y3M", "0Y6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]
for mat in matList:
    models.append(
        AutoReg(
            endog = trainSplit[mat]["train"][mat],
            lags = 5,
            trend = "c"
        )
    )
forecasts = []
for model in models:
    forecasts.append(model.fit().forecast(steps=20).to_numpy())

#print(forecasts)

horizons = []
errors = []
rmse = []
for i in range(len(forecasts)):
    horizons.append([])
    errors.append([])
    rmse.append([])

    horizons[i].extend((forecasts[i][0], forecasts[i][4], forecasts[i][19]))
    errors[i].extend((forecasts[i][0] - testArr[i][0], forecasts[i][4] - testArr[i][4], forecasts[i][19] - testArr[i][19]))
    rmse[i].extend((np.sqrt(np.mean(errors[i][0]**2)), np.sqrt(np.mean(errors[i][1]**2)), np.sqrt(np.mean(errors[i][2]**2))))

    print("Maturity: ", matList[i], "\nRMSE for 1 Day forecast: ", rmse[i][0], "\nRMSE for 5 day forecast: ", rmse[i][1], "\nRMSE for 20 day forecast: ", rmse[i][2], "\n\n")    






Maturity:  0Y1M 
RMSE for 1 Day forecast:  0.0074312867184840314 
RMSE for 5 day forecast:  0.011214294928832302 
RMSE for 20 day forecast:  0.03715056511464532 


Maturity:  0Y3M 
RMSE for 1 Day forecast:  0.00028915025096676317 
RMSE for 5 day forecast:  0.01747948066609606 
RMSE for 20 day forecast:  0.03876222873265078 


Maturity:  0Y6M 
RMSE for 1 Day forecast:  0.011186367117335116 
RMSE for 5 day forecast:  7.724510208007018e-05 
RMSE for 20 day forecast:  0.08380466802066966 


Maturity:  1Y 
RMSE for 1 Day forecast:  0.03800189603802817 
RMSE for 5 day forecast:  0.03585117908560953 
RMSE for 20 day forecast:  0.15155943409307948 


Maturity:  2Y 
RMSE for 1 Day forecast:  0.05993654256203884 
RMSE for 5 day forecast:  0.0645411474878399 
RMSE for 20 day forecast:  0.1897758495222308 


Maturity:  3Y 
RMSE for 1 Day forecast:  0.05059820863670961 
RMSE for 5 day forecast:  0.0529227943826629 
RMSE for 20 day forecast:  0.1562213121315117 


Maturity:  5Y 
RMSE for 1 Day forec